# Bonus Act — A Minimal Coding Agent

Everything so far generalizes. Give the same `Agent` class filesystem and shell tools, and it becomes a coding agent — the same shape as Claude Code, Cursor, and Aider. This is loosely modeled on [nanocode](https://github.com/1rgs/nanocode), a ~200-line reference coding agent, adapted to our style.

In [ ]:
import sys, inspect
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv(override=True)

from mini_agent import Agent, read_file, write_file, edit_file, glob_files, grep_files, bash
from mini_agent.coding_tools import WORKSPACE_DIR

## Six tools, ~150 lines
`read_file`, `write_file`, `edit_file`, `glob_files`, `grep_files`, `bash` — nanocode's exact toolset, in `mini_agent`'s plain-function style: type hints and a docstring, nothing else. One deliberate difference from the reference: every tool but `bash` is scoped to a `workspace/` directory — path escapes are rejected — so a live demo can't wander outside it. `bash` can still do anything the presenter's shell user can; same caveat as `execute_python` in Act 3, this is a teaching tool, not a sandbox.

In [ ]:
print(inspect.getdoc(edit_file))
print(inspect.signature(edit_file))

## Build the coding agent
A system prompt telling it where it's working, and the six tools — that's the whole difference from every other agent in this workshop.

In [ ]:
coding_agent = Agent(
    tools=[read_file, write_file, edit_file, glob_files, grep_files, bash],
    instructions=f"You are a concise coding assistant. Workspace: {WORKSPACE_DIR}",
    max_steps=8,
)

## Give it a real task

In [ ]:
answer = await coding_agent.run(
    "Write a Python script fizzbuzz.py that prints FizzBuzz from 1 to 20, then run it and show me the output."
)
print("\nFINAL:", answer)

It wrote the file, ran it, read the output, and reported back — all on its own. If the first `bash` call fails for any reason (wrong interpreter name, a typo), watch what happens next: it's just another tool result, and the agent adapts on the next step instead of crashing.

## Read, search, and edit
`agent.messages` still remembers everything from the task above — ask it to change what it just wrote:

In [ ]:
answer2 = await coding_agent.run(
    "List the files in the workspace, read fizzbuzz.py, then edit it to only go up to 15 instead of 21 "
    "(the range upper bound), then run it again to confirm.",
    verbose=True,
)
print("\nFINAL:", answer2)

## A CLI you can actually chat with
Everything above ran one task per cell. `mini_agent/cli.py` wraps the exact same tools and instructions in a loop you can run from an actual terminal — closer to how you'd really use something like Claude Code. Here's the whole loop:

In [ ]:
import inspect
from mini_agent import cli
print(inspect.getsource(cli.repl))

Open a terminal in this `workshop/` folder — not this notebook — and run:

```bash
python3 -m mini_agent.cli
```

Then just talk to it: ask it to write a script, read it back, fix a bug, run it again. It remembers the whole conversation, exactly like Act 3 — same `Agent`, same `.messages` list, just wrapped in an `input()` loop instead of separate cells. Type `exit` or `quit` (or Ctrl+C) to leave.

## Wrap-up
This is genuinely most of what production coding agents do under the hood: an LLM, a loop, and a handful of file/shell tools — the same `Agent.run` from Act 2, unmodified. What separates a demo like this from Claude Code, Cursor, or Aider isn't a different core loop, it's engineering *around* it: permission prompts before destructive actions, real sandboxing (not just a scoped directory), diffing UI, multi-file context management, checkpointing.

**A note on safety, if you build on this after the workshop:** `read_file`/`write_file`/`edit_file`/`glob_files`/`grep_files` reject path escapes, but `bash` does not — a determined (or confused) agent can still run anything your shell user can. Don't point this at anything you're not prepared to lose.